# 18과: 암호화 영수증으로 AI 에이전트 보안 강화

## 실습 노트북

이 노트북에서는 네 가지 실습을 진행합니다:

1. 에이전트 도구 호출에 대한 **첫 영수증 서명** 및 검증.
2. **영수증 변조** 및 검증 실패 확인.
3. **세 개의 영수증 체인 구축** 및 체인 무결성 확인.
4. Microsoft Agent Framework 도구 호출을 **랩핑하여 모든 작업에 영수증 생성**.

모든 암호화 원시 기능은 잘 관리되는 라이브러리에서 가져옵니다 (`pynacl`는 Ed25519용, `jcs`는 RFC 8785 정식 JSON용, `hashlib`는 파이썬 표준 라이브러리의 SHA-256용). 영수증 로직 자체는 읽고 수정할 수 있는 평범한 파이썬입니다.

셀을 순서대로 실행하세요. 각 섹션은 짧고 독립적입니다.


## 설정

두 가지 의존성을 설치합니다. 두 라이선스 모두 허용적인 라이선스입니다(Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## 도우미 유틸리티

이 두 도우미는 base64url 인코딩(패딩 없이)과 임의 객체의 SHA-256 해싱을 처리합니다. 이들은 나머지 노트북이 영수증 로직 자체에 집중할 수 있도록 해줍니다.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## 섹션 1: 첫 번째 영수증에 서명하기

<strong>Contoso Travel</strong>의 에이전트가 고객을 위해 시드니에서 로스앤젤레스로 가는 항공편을 조회했다고 상상해 보세요. 향후 감사자가 별도로 고정된 발행자 키 아래에서 이를 검증할 수 있도록 이 도구 호출에 대한 게이트웨이의 진술을 기록하려고 합니다.

### 1.1단계: 서명 키 생성

실제 환경에서는 에이전트의 서명 키가 하드웨어 보안 모듈(HSM), Azure Key Vault 또는 이와 유사한 보호된 저장소에 보관됩니다. 이 강의에서는 메모리에서 새 키를 생성합니다.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### 1.2단계: 영수증 페이로드 빌드

페이로드에는 영수증이 증명해야 하는 모든 내용이 포함됩니다: 누가 행동했는지, 어떤 도구를 사용했는지, 어떤 인수가 있었는지, 무엇이 반환되었는지, 어떤 정책 하에서, 그리고 언제였는지. 인수와 결과를 인라인으로 포함하는 대신 해시하여 영수증이 민감한 내용을 노출하지 않도록 합니다.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### 1.3단계: 영수증 서명 및 조립

세 단계:

1. JCS를 사용하여 페이로드를 정규화하여 동일한 논리적 영수증을 생성하는 두 구현이 바이트가 동일하도록 합니다.
2. Ed25519 개인 키로 정규화된 바이트를 직접 서명합니다. PureEdDSA는 메시지를 내부적으로 해시하므로, 추가 사전 해시는 프로토콜을 변경하게 됩니다.

그런 다음 서명을 원본 페이로드에 첨부하여 최종 영수증을 만듭니다.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### 1.4단계: 영수증 검증

검증은 과정을 역으로 수행합니다. 서명을 떼어내고, 표준 바이트를 다시 계산한 다음, 발급자의 공개 키로 서명을 확인합니다.

검증은 오프라인으로 수행됩니다: 호출할 서비스도 없고 네트워크 접근도 없습니다. 그러나 영수증 외에 하나가 더 필요합니다: 심사관이 이미 신뢰하는 경로를 통해 얻은 발급자의 공개 키로, 발급자가 공개한 JWK 세트나 사전에 고정한 값 등이 그 예입니다. 영수증 안의 `public_key` 필드는 그 역할을 할 수 없습니다. 왜냐하면 영수증을 작성하는 사람이 그 필드도 같이 작성하기 때문입니다. 2장에서는 이것이 방지하는 위조 사례를 보여줍니다.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


`Receipt is valid: True`, `Pre-hashed receipt valid: False` 및 `Malformed receipts rejected: True`를 확인할 수 있어야 합니다. 긍정 사례는 direct-JCS 서명 경로가 작동함을 증명하고; 부정 대조는 서명-범위 규칙을 산문으로 남기지 않고 실행 가능하게 만들며; 마지막 줄은 예외를 발생시키는 대신 잘못된 입력이 `False`를 반환하는 것을 보여줍니다.


## 섹션 2: 영수증 조작하기

영수증의 핵심 목적은 변조 여부를 알 수 있게 하는 것입니다. 이를 증명해봅시다.

영수증의 문자를 단 한 개만 수정하고 검증이 실패하는 것을 지켜볼 것입니다.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### What just happened?

When we changed `policy_id`, the canonical bytes changed. The signature (which was over the original canonical bytes) no longer matches. Verification correctly returns `False`.

There is no way to change a signed field, meaning anything outside the `signature` object, and still have the receipt verify against the issuer's public key, unless the attacker has the issuer's private key. The `signature` object itself is not signed: `sig` is the signature, and `verify_receipt` ignores `public_key`, so editing that field alone changes nothing. As long as the private key is in a key vault and verifiers use the issuer's independently published key, a change to a signed field is impossible to hide. That condition matters: the next cell shows a modified receipt that verifies perfectly against a different key.

Try it yourself: modify the `tool_name` or `agent_id` or `timestamp` in the cell above and re-run. Every change to a signed field produces an invalid receipt.


### 서명이 잡아내지 못하는 변조

위의 공격자는 다시 서명하지 않았습니다. 좀 더 신중한 공격자는: 영수증을 변경하고,
스스로 생성한 키로 서명한 다음, 그 키를
`public_key` 필드에 넣을 것입니다. 결과는 내용에 대해
서명이 완전히 진짜인 영수증입니다.

이것을 실행하고 두 검사 중 어떤 검사가 이를 잡아내는지 보십시오.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


`True` 그리고 `False`를 보셔야 합니다. 세 개의 `assert` 줄이 이 부분을
검사하게 만듭니다: 셀은 위조물이 발행자의 키에 대해 검증되면 오류를 발생시킵니다.


위조물은 내부적으로 일관됩니다. 서명은 실제로 내용과
그가 가진 키와 일치하므로, 영수증에서 키를 읽는 검증기는 이를 유효하다고 보고합니다.
영수증에는 잘못된 점이 없어 보이는데, 왜냐하면
아무 것도 일관성이 없지 않기 때문입니다.

유일하게 검출하는 것은 공격자가 영향을 줄 수 없는 키에 대해 검사하는 것입니다.
이것이 `verify_receipt`가 키를 인수로 받는 이유입니다: 
`verify_receipt(receipt)` 서명을 가진 함수는 안전하게 작성할 수 없습니다,
왜냐하면 접근할 수 있는 유일한 키가 공격자가 제공한 키이기 때문입니다.

같은 영수증이 노트북 밖에서 검사하고 싶을 경우 `sample_receipts/04_resigned_receipt.json`로 저장됩니다.
노트북 밖에서 그것을 검사하고 싶습니다.


## 섹션 3: 영수증을 서로 연결하기

하나의 영수증은 한 가지 동작을 보호합니다. 대부분의 에이전트는 여러 동작을 수행합니다. 전체 시퀀스를 변조 증명하려면 각 영수증에 이전 영수증의 해시를 포함하여 새 영수증의 페이로드에 이전 영수증과 연결합니다.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

만약 누군가가 제시된 시퀀스 내에서 영수증을 제거하거나 순서를 바꾸면 다음 링크가 끊어집니다. 잘림(truncation)이나 서명자가 제어하는 접미사 재작성(suffix rewriting) 감지 또한 고정된 현재 헤드, 외부 체크포인트, 또는 인증된 인구 집단 커밋을 필요로 합니다.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


이제 중간 영수증을 변조하여 체인을 끊고 다시 검증해 보세요. 변조된 영수증은 서명 검증에 실패하고, 다음 영수증은 체인 링크 검증에 실패합니다(왜냐하면 그 `previous_receipt_hash` 가 더 이상 수정된 중간 영수증의 해시와 일치하지 않기 때문입니다).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


영수증 0은 여전히 검증됩니다(변경되지 않았으며 종속되어야 할 이전 영수증이 없습니다). 영수증 1은 `tool_args_hash`를 변경했기 때문에 서명 확인에 실패합니다. 영수증 2는 체인 연결 확인에 실패하는데, 이는 `previous_receipt_hash`가 원본(현재 변경된) 영수증 1을 기준으로 계산되었기 때문입니다.

공격자가 수정된 영수증 1에 다시 서명하더라도, 발급자의 키를 갖고 있지 않으면 새로운 서명이 발급자의 키에 대해 실패합니다(자신의 키로 재서명하는 것은 2절에서 보여준 바와 같이 실패합니다). 그리고 영수증 2의 체인 연결 불일치도 변조를 드러냅니다. 변경을 숨기려면 공격자는 수정 지점 이후 모든 영수증에 다시 서명해야 하며, 이는 발급자의 개인 키를 소유해야 가능한 일입니다.


## 섹션 4: 영수증 서명으로 에이전트 도구 호출 래핑하기

실제 배포에서는 모든 에이전트 작성자가 `make_receipt` 를 호출하는 것을 기억하게 하고 싶지 않습니다. 모든 도구 호출에 대해 영수증 서명이 자동으로 이루어지길 원합니다.

가장 간단한 패턴은, 호출 가능한 도구 함수를 받아서 영수증을 출력하는 버전으로 반환하는 래퍼 클래스입니다. 이는 Microsoft 에이전트 프레임워크(`agent_framework.foundry`)를 포함한 어떤 에이전트 프레임워크에도 적용할 수 있습니다.

Microsoft Foundry 프로젝트가 설정되어 있지 않아도, 아래의 로컬 목(mock)이 여전히 이 패턴을 보여줍니다.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Microsoft Agent Framework와 통합하기

위의 `ReceiptedTool` 래퍼는 프레임워크에 구애받지 않습니다. Microsoft Agent Framework로 구축된 에이전트 내에서 사용하려면 래핑된 함수를 도구로 등록하세요. 개요(목업을 실제 Microsoft Foundry 도구 등록으로 교체할 수 있음):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

에이전트 프레임워크는 영수증에 대해 아무것도 알 필요가 없습니다. 영수증 서명은 도구 주변에 래핑되어 있으며 프레임워크에 고정되지 않습니다. 이렇게 하면 에이전트를 다시 작성하지 않고 기존 에이전트 코드에 출처를 추가할 수 있습니다.


## 요약 및 도전 과제

여러분은:

- Ed25519 키 쌍을 생성했습니다.
- 에이전트 도구 호출에 대한 영수증을 생성하고 서명했습니다.
- 영수증과 무관하게 독립적으로 얻은 발급자의 공개키만 사용하여 오프라인에서 영수증을 검증했습니다.
- 영수증을 변조하고 검증 실패를 관찰했습니다.
- 세 개의 영수증으로 해시 체인 시퀀스를 작성했습니다.
- 체인의 중간을 변조하고 서명 실패와 체인 링크 실패를 모두 관찰했습니다.
- 도구 함수에 자동 영수증 서명 기능을 래핑했습니다.

**도전 과제.** 영수증 스키마에 `request_id` 필드(분산 추적을 위한 UUID)를 확장하세요. `make_receipt`을 업데이트하여 포함시키고, 영수증이 끝까지 검증되는지 확인하세요. 그런 다음 서명 후 필드를 수정하고 검증 실패를 확인하세요. 이렇게 하면 정식 인코딩의 모든 바이트가 서명에 어떻게 기여하는지 내면화할 수 있습니다.

**중요한 경계.** 고정된 키와 체인 컨텍스트 하에서 영수증은 서명 귀속, 내용 무결성, 제시된 전임자와의 링크를 증명합니다. 그러나 실행, 출처 진실성, 신뢰된 시간, 완전성, 정책 평가 또는 정확성은 증명하지 않습니다. 영수증은 기반입니다. 거버넌스는 그 위에 구축하는 시스템입니다.

이 경계를 염두에 두고 레슨 README를 다시 읽어보세요. 팀이 영수증과 관련해 가장 자주 하는 실수는 “우리에겐 영수증이 있다”가 “우리가 거버넌스되고 있다”를 의미한다고 착각하는 것입니다. 그렇게 되지 않습니다. 영수증은 에이전트 행동을 감사를 가능하게 합니다. 그것이 행동을 옳게 만드는 것은 아닙니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
